In [1]:
import sounddevice as sd
import numpy as np
import wave
import io
import whisper
from googletrans import Translator
import speech_recognition as sr
import edge_tts
import asyncio
from IPython.display import Audio, display

# ---------------------------- Voice Map ---------------------------- #
voice_map = {
    'english': {
        'male': 'en-US-GuyNeural',
        'female': 'en-US-JennyNeural'
    },
    'french': {
        'male': 'fr-FR-HenriNeural',
        'female': 'fr-FR-DeniseNeural'
    },
    'german': {
        'male': 'de-DE-ConradNeural',
        'female': 'de-DE-KatjaNeural'
    },
    'spanish': {
        'male': 'es-ES-AlvaroNeural',
        'female': 'es-ES-ElviraNeural'
    },
    'hindi': {
        'male': 'hi-IN-AaravNeural',
        'female': 'hi-IN-SwaraNeural'
    },
    'urdu': {
        'male': 'ur-PK-AsadNeural',
        'female': 'ur-PK-UzmaNeural'
    },
    'telugu': {
        'male': 'te-IN-MohanNeural',
        'female': 'te-IN-ShrutiNeural'
    }
}

# ---------------------------- Get User Input ---------------------------- #
dest_language_input = input("Enter the language you want to translate to (e.g., English, French): ").lower()
gender = input("Do you want a male or female voice? ").lower()

if dest_language_input not in voice_map:
    raise ValueError(f"Unsupported language '{dest_language_input}'.")

if gender not in voice_map[dest_language_input]:
    raise ValueError(f"Unsupported gender '{gender}' for language '{dest_language_input}'.")

voice = voice_map[dest_language_input][gender]

# ---------------------------- Record Audio ---------------------------- #
def record_audio():
    samplerate = 16000
    print("Press Enter to start recording...")
    input()
    print("Recording... Press Enter again to stop.")
    
    audio_data = []

    def callback(indata, frames, time, status):
        audio_data.append(indata.copy())

    stream = sd.InputStream(callback=callback, channels=1, samplerate=samplerate)
    stream.start()
    input()  # Wait for Enter to stop
    stream.stop()
    print("Recording stopped.")

    audio_np = np.concatenate(audio_data, axis=0)
    byte_io = io.BytesIO()
    with wave.open(byte_io, 'wb') as wf:
        wf.setnchannels(1)
        wf.setsampwidth(2)
        wf.setframerate(samplerate)
        wf.writeframes((audio_np * 32767).astype(np.int16).tobytes())
    byte_io.seek(0)
    return byte_io

audio_bytes_io = record_audio()

# Save to temp WAV for Whisper and SR
temp_wav_path = "temp.wav"
with wave.open(audio_bytes_io, 'rb') as wf:
    with wave.open(temp_wav_path, 'wb') as temp:
        temp.setnchannels(wf.getnchannels())
        temp.setsampwidth(wf.getsampwidth())
        temp.setframerate(wf.getframerate())
        temp.writeframes(wf.readframes(wf.getnframes()))

# ---------------------------- Detect Language using Whisper ---------------------------- #
model = whisper.load_model("tiny")
result = model.transcribe(temp_wav_path)
detected_lang_code = result['language']
print(f"Detected language: {detected_lang_code}")

# ---------------------------- Transcribe ---------------------------- #
if detected_lang_code == 'en':
    print("Transcribing with Whisper (English)...")
    result_en = model.transcribe(temp_wav_path, language='en')
    segments = result_en['segments']
    transcript = "".join([seg['text'] for seg in segments])

else:
    print("Transcribing with SpeechRecognition (non-English)...")
    recognizer = sr.Recognizer()
    with sr.AudioFile(temp_wav_path) as source:
        audio = recognizer.record(source)
    try:
        transcript = recognizer.recognize_google(audio, language=detected_lang_code)
    except sr.UnknownValueError:
        transcript = "Google Speech Recognition could not understand audio"
    except sr.RequestError as e:
        transcript = f"Could not request results from Google Speech Recognition service; {e}"

print(f"\nOriginal Transcribed Text:\n{transcript}")

# ---------------------------- Translate ---------------------------- #
translator = Translator()
try:
    translated = translator.translate(transcript, dest=dest_language_input)
    translated_text = translated.text
except Exception as e:
    translated_text = f"Translation failed: {e}"

print(f"\nTranslated Text ({dest_language_input.title()}):\n{translated_text}")

# ---------------------------- Speak (Edge TTS) ---------------------------- #
async def speak_text(text, voice):
    communicate = edge_tts.Communicate(text=text, voice=voice)
    audio_data = bytearray()

    async for message in communicate.stream():
        if message["type"] == "audio":
            audio_data += message["data"]

    audio_bytes = io.BytesIO(audio_data)
    display(Audio(audio_bytes.read(), autoplay=True))

await speak_text(translated_text, voice)

# ---------------------------- Return Translated Text ---------------------------- #
translated_text  # <--- ✅ This makes it the notebook's final output


Enter the language you want to translate to (e.g., English, French):  english
Do you want a male or female voice?  male


Press Enter to start recording...


Recording... Press Enter again to stop.


Recording stopped.
Detected language: en
Transcribing with Whisper (English)...

Original Transcribed Text:


Translated Text (English):
Translation failed: the JSON object must be str, bytes or bytearray, not NoneType


ClientConnectorCertificateError: Cannot connect to host api.msedgeservices.com:443 ssl:True [SSLCertVerificationError: (1, '[SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: certificate has expired (_ssl.c:1147)')]

In [ ]:
import multiprocessing as mp
import asyncio
import time
import queue
import numpy as np
import torch
import sounddevice as sd
import soundfile as sf

from faster_whisper import WhisperModel
from googletrans import Translator
import speech_recognition as sr
import edge_tts

from IPython.display import display, Audio, clear_output

SAMPLE_RATE = 16000
CHANNELS = 1
CHUNK_DURATION = 0.8
CHUNK_SAMPLES = int(SAMPLE_RATE * CHUNK_DURATION)
OVERLAP_DURATION = 0.2
OVERLAP_SAMPLES = int(SAMPLE_RATE * OVERLAP_DURATION)

def choose_edge_voice(lang: str, gender: str) -> str:
    if lang.startswith("en"):
        return "en-US-JennyNeural" if gender.lower().startswith("f") else "en-US-GuyNeural"
    if lang.startswith("fr"):
        return "fr-FR-DeniseNeural" if gender.lower().startswith("f") else "fr-FR-HenriNeural"
    if lang.startswith("hi"):
        return "hi-IN-SwaraNeural" if gender.lower().startswith("f") else "hi-IN-RaviNeural"
    return "en-US-JennyNeural"

def audio_capture_proc(audio_queue: mp.Queue, stop_event: mp.Event):
    def callback(indata, frames, time_info, status):
        if stop_event.is_set():
            raise sd.CallbackAbort
        mono = indata.mean(axis=1)
        pcm = (mono * np.iinfo(np.int16).max).astype(np.int16)
        audio_queue.put(pcm.tobytes())

    with sd.InputStream(samplerate=SAMPLE_RATE, channels=CHANNELS,
                        blocksize=CHUNK_SAMPLES, callback=callback):
        while not stop_event.is_set():
            time.sleep(0.1)
    print("[Audio] capture ended.")

def transcribe_proc(audio_queue: mp.Queue, text_queue: mp.Queue, stop_event: mp.Event):
    whisper = WhisperModel("small", device="cuda" if torch.cuda.is_available() else "cpu", compute_type="int8")
    recognizer = sr.Recognizer()

    buf = bytearray()
    buf_samples = 0

    while not stop_event.is_set():
        try:
            chunk_bytes = audio_queue.get(timeout=0.5)
        except queue.Empty:
            continue

        buf.extend(chunk_bytes)
        buf_samples += CHUNK_SAMPLES

        if buf_samples >= CHUNK_SAMPLES + OVERLAP_SAMPLES:
            pcm = np.frombuffer(buf, dtype=np.int16).astype(np.float32) / np.iinfo(np.int16).max
            segments, info = whisper.transcribe(pcm, beam_size=5, language=None,
                                                vad_filter=True, condition_on_previous_text=True)
            src_lang = info.language
            text = "".join([seg.text for seg in segments]).strip()

            tail_bytes = buf[-OVERLAP_SAMPLES * 2:]
            buf = bytearray(tail_bytes)
            buf_samples = len(buf) // 2

            if not text and src_lang != "en":
                try:
                    audio_data = sr.AudioData(chunk_bytes, SAMPLE_RATE, 2)
                    text = recognizer.recognize_google(audio_data, language=src_lang)
                except Exception:
                    text = ""

            if text:
                text_queue.put((src_lang, text))
    print("[Transcription] ended.")

def translate_proc(text_queue: mp.Queue, translated_queue: mp.Queue, target_lang: str, stop_event: mp.Event):
    translator = Translator()
    while not stop_event.is_set():
        try:
            src_lang, text = text_queue.get(timeout=0.5)
        except queue.Empty:
            continue
        if not text.strip():
            continue
        try:
            res = translator.translate(text, src=src_lang, dest=target_lang)
            translated_queue.put((text, res.text))
        except Exception as e:
            print("[Translate] error:", e)
    print("[Translation] ended.")

async def edge_tts_stream(text, voice):
    communicate = edge_tts.Communicate(text, voice=voice)
    audio_bytes = b""
    async for chunk in communicate.stream():
        audio_bytes += chunk
    return audio_bytes

def main():
    target_lang = input("Enter target (output) language code, e.g. 'en', 'fr', 'hi': ").strip()
    gender = input("Enter target gender (male / female): ").strip()

    DURATION = 10

    audio_queue = mp.Queue(maxsize=20)
    text_queue = mp.Queue(maxsize=20)
    translated_queue = mp.Queue(maxsize=20)
    stop_event = mp.Event()

    p_cap = mp.Process(target=audio_capture_proc, args=(audio_queue, stop_event))
    p_trans = mp.Process(target=transcribe_proc, args=(audio_queue, text_queue, stop_event))
    p_transl = mp.Process(target=translate_proc, args=(text_queue, translated_queue, target_lang, stop_event))

    p_cap.start()
    p_trans.start()
    p_transl.start()

    print(f"Recording and translating for {DURATION} seconds...")

    start_time = time.time()
    voice = choose_edge_voice(target_lang, gender)

    try:
        while time.time() - start_time < DURATION:
            # Try to get translation results from queue
            try:
                original_text, translated_text = translated_queue.get(timeout=0.5)
            except queue.Empty:
                continue

            clear_output(wait=True)
            print(f"Transcribed: {original_text}")
            print(f"Translated: {translated_text}")

            # Synthesize and play audio inline
            audio_bytes = asyncio.run(edge_tts_stream(translated_text, voice))
            display(Audio(audio_bytes, rate=24000))  # edge-tts audio sample rate ~24kHz

    except KeyboardInterrupt:
        print("Interrupted by user")

    stop_event.set()

    p_cap.join()
    p_trans.join()
    p_transl.join()

    print("Exited cleanly.")

if __name__ == "__main__":
    main()
